In [ ]:
# ============================================================
# CTI PROJECT — EVERY CODE LINE COMMENTED FOR STUDY
# ============================================================
# This is a study/defense version of the uploaded CTI program.
# Original code is preserved line-by-line, with an explanation comment added
# to every non-empty executable line. The comments explain what each line
# is doing in the context of the MITRE ATT&CK threat-group relationship project.
# ============================================================

import pandas as pd  # Line 1: Imports pandas for reading and manipulating tabular data.
import numpy as np  # Line 2: Imports NumPy for numerical and random-number operations.

from sklearn.ensemble import RandomForestClassifier  # Line 4: Imports the Random Forest classification algorithm.
from sklearn.model_selection import train_test_split  # Line 5: Imports the function used to split data into training and testing sets.
from sklearn.metrics import (  # Line 6: Starts importing the evaluation metrics used to measure model performance.
    accuracy_score,  # Line 7: Imports this specific classification metric.
    precision_score,  # Line 8: Imports this specific classification metric.
    recall_score,  # Line 9: Imports this specific classification metric.
    f1_score  # Line 10: Imports this specific classification metric.
)  # Line 11: Closes the current function/import/configuration block.

# ============================================================
# CONFIGURATION
# ============================================================

FILE = "MITRE_Integrated_Master_Dataset.csv"  # Line 17: Stores the integrated MITRE dataset filename in one reusable variable.

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def normalize(text):  # Line 23: Defines a helper function that standardizes text before matching.
    """  # Line 24: Defines or formats a text string used by the program.
    Normalize text for reliable matching.  # Line 25: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    """  # Line 26: Defines or formats a text string used by the program.
    if pd.isna(text):  # Line 27: Checks whether the current value is missing.
        return ""  # Line 28: Returns an empty string when no text is available.

    text = str(text).strip().lower()  # Line 30: Converts the value to text, removes surrounding spaces, and changes it to lowercase.

    # Replace multiple spaces
    text = " ".join(text.split())  # Line 33: Collapses repeated whitespace into single spaces.

    return text  # Line 35: Returns the cleaned text.


def split_values(value):  # Line 38: Defines a helper function for splitting semicolon-separated values.
    """  # Line 39: Defines or formats a text string used by the program.
    Split semicolon-separated ATT&CK values.  # Line 40: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    """  # Line 41: Defines or formats a text string used by the program.
    if pd.isna(value):  # Line 42: Checks whether the current value is missing.
        return []  # Line 43: Builds and returns a cleaned list of values.

    return [  # Line 45: Builds and returns a cleaned list of values.
        x.strip()  # Line 46: This line performs the operation shown in the original code; the surrounding block determines its exact role.
        for x in str(value).split(";")  # Line 47: Splits one cell into separate values using semicolons.
        if x.strip()  # Line 48: Checks a condition and executes the indented block when it is true.
    ]  # Line 49: This line performs the operation shown in the original code; the surrounding block determines its exact role.


# ============================================================
# 1. LOAD DATASET
# ============================================================

print("\nLoading integrated dataset...")  # Line 56: Prints information to the console for the user or for debugging.

data = pd.read_csv(FILE)  # Line 58: Loads the integrated CSV dataset into a pandas DataFrame.

print("\nDataset Shape:", data.shape)  # Line 60: Prints information to the console for the user or for debugging.

print("\nColumns:")  # Line 62: Prints information to the console for the user or for debugging.
print(list(data.columns))  # Line 63: Prints information to the console for the user or for debugging.


# ============================================================
# 2. CLEAN DATA
# ============================================================

required_columns = [  # Line 70: Defines the columns that must exist for the program to work.
    "group_id",  # Line 71: Defines or formats a text string used by the program.
    "group_name",  # Line 72: Defines or formats a text string used by the program.
    "technique_id",  # Line 73: Defines or formats a text string used by the program.
    "technique_name"  # Line 74: Defines or formats a text string used by the program.
]  # Line 75: This line performs the operation shown in the original code; the surrounding block determines its exact role.

missing_columns = [  # Line 77: Builds a list of required columns that are absent from the dataset.
    col for col in required_columns  # Line 78: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    if col not in data.columns  # Line 79: Checks a condition and executes the indented block when it is true.
]  # Line 80: This line performs the operation shown in the original code; the surrounding block determines its exact role.

if missing_columns:  # Line 82: Checks whether any required dataset columns are missing.

    print("\nERROR: Missing columns:")  # Line 84: Prints information to the console for the user or for debugging.
    print(missing_columns)  # Line 85: Prints information to the console for the user or for debugging.

    raise SystemExit  # Line 87: Stops the program because the required data structure is not available.


# Remove duplicates
data = data.drop_duplicates(  # Line 91: Removes duplicate group-technique relationships.
    subset=[  # Line 92: Tells pandas which columns define a duplicate relationship.
        "group_id",  # Line 93: Defines or formats a text string used by the program.
        "technique_id"  # Line 94: Defines or formats a text string used by the program.
    ]  # Line 95: This line performs the operation shown in the original code; the surrounding block determines its exact role.
).copy()  # Line 96: This line performs the operation shown in the original code; the surrounding block determines its exact role.


# Remove rows without group/technique
data = data.dropna(  # Line 100: Removes rows that do not have a group ID or technique ID.
    subset=[  # Line 101: Tells pandas which columns define a duplicate relationship.
        "group_id",  # Line 102: Defines or formats a text string used by the program.
        "technique_id"  # Line 103: Defines or formats a text string used by the program.
    ]  # Line 104: This line performs the operation shown in the original code; the surrounding block determines its exact role.
).copy()  # Line 105: This line performs the operation shown in the original code; the surrounding block determines its exact role.


print(  # Line 108: Prints information to the console for the user or for debugging.
    "\nAfter cleaning:",  # Line 109: Defines or formats a text string used by the program.
    data.shape  # Line 110: This line performs the operation shown in the original code; the surrounding block determines its exact role.
)  # Line 111: Closes the current function/import/configuration block.


# ============================================================
# 3. ML FEATURES
# ============================================================

feature_columns = [  # Line 118: Defines the six numerical features used by the Random Forest model.
    "group_technique_count",  # Line 119: Defines or formats a text string used by the program.
    "technique_frequency",  # Line 120: Defines or formats a text string used by the program.
    "group_software_count",  # Line 121: Defines or formats a text string used by the program.
    "group_campaign_count",  # Line 122: Defines or formats a text string used by the program.
    "software_support_count",  # Line 123: Defines or formats a text string used by the program.
    "campaign_support_count"  # Line 124: Defines or formats a text string used by the program.
]  # Line 125: This line performs the operation shown in the original code; the surrounding block determines its exact role.


# Convert features to numbers
for column in feature_columns:  # Line 129: Loops through every ML feature so it can be converted to numeric form.

    data[column] = pd.to_numeric(  # Line 131: Converts the selected feature values into numbers.
        data[column],  # Line 132: This line performs the operation shown in the original code; the surrounding block determines its exact role.
        errors="coerce"  # Line 133: Turns values that cannot be converted into numbers into missing values.
    ).fillna(0)  # Line 134: Replaces missing numerical feature values with zero.


# ============================================================
# 4. POSITIVE DATA
# ============================================================

positive = data[  # Line 141: Creates the positive examples from documented MITRE group-technique relationships.
    [  # Line 142: This line performs the operation shown in the original code; the surrounding block determines its exact role.
        "group_id",  # Line 143: Defines or formats a text string used by the program.
        "technique_id",  # Line 144: Defines or formats a text string used by the program.
        *feature_columns  # Line 145: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    ]  # Line 146: This line performs the operation shown in the original code; the surrounding block determines its exact role.
].copy()  # Line 147: This line performs the operation shown in the original code; the surrounding block determines its exact role.

positive["label"] = 1  # Line 149: Labels documented relationships as class 1, meaning positive.


# ============================================================
# 5. NEGATIVE DATA
# ============================================================

print("\nCreating negative samples...")  # Line 156: Prints information to the console for the user or for debugging.

rng = np.random.default_rng(42)  # Line 158: Creates a reproducible random-number generator using seed 42.

all_techniques = (  # Line 160: Collects every unique technique ID available in the integrated dataset.
    data["technique_id"]  # Line 161: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    .dropna()  # Line 162: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    .unique()  # Line 163: This line performs the operation shown in the original code; the surrounding block determines its exact role.
)  # Line 164: Closes the current function/import/configuration block.


negative_rows = []  # Line 167: Creates an empty list to store constructed negative examples.


for group_id in data["group_id"].unique():  # Line 170: Processes each unique threat group one at a time.

    group_data = data[  # Line 172: Filters the dataset to records belonging to the current threat group.
        data["group_id"] == group_id  # Line 173: Assigns the value on the right-hand side to the variable on the left-hand side.
    ]  # Line 174: This line performs the operation shown in the original code; the surrounding block determines its exact role.

    used_techniques = set(  # Line 176: Gets the techniques already documented for the current group.
        group_data["technique_id"]  # Line 177: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    )  # Line 178: Closes the current function/import/configuration block.

    unused_techniques = [  # Line 180: Finds techniques not documented for the current group to create a sampled negative pair.
        technique  # Line 181: This line performs the operation shown in the original code; the surrounding block determines its exact role.
        for technique in all_techniques  # Line 182: Starts a loop that processes each item in the specified iterable.
        if technique not in used_techniques  # Line 183: Checks a condition and executes the indented block when it is true.
    ]  # Line 184: This line performs the operation shown in the original code; the surrounding block determines its exact role.

    if not unused_techniques:  # Line 186: Checks whether there are any unused techniques available for sampling.
        continue  # Line 187: Skips this group and moves to the next group.

    selected_technique = rng.choice(  # Line 189: Randomly selects one technique that is not documented for this group.
        unused_techniques  # Line 190: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    )  # Line 191: Closes the current function/import/configuration block.

    group_software_count = (  # Line 193: Reads the group-level software count for the current group.
        group_data[  # Line 194: This line performs the operation shown in the original code; the surrounding block determines its exact role.
            "group_software_count"  # Line 195: Defines or formats a text string used by the program.
        ].iloc[0]  # Line 196: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    )  # Line 197: Closes the current function/import/configuration block.

    group_campaign_count = (  # Line 199: Reads the group-level campaign count for the current group.
        group_data[  # Line 200: This line performs the operation shown in the original code; the surrounding block determines its exact role.
            "group_campaign_count"  # Line 201: Defines or formats a text string used by the program.
        ].iloc[0]  # Line 202: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    )  # Line 203: Closes the current function/import/configuration block.

    negative_rows.append({  # Line 205: Adds one constructed negative group-technique example to the list.

        "group_id": group_id,  # Line 207: Defines or formats a text string used by the program.

        "technique_id": selected_technique,  # Line 209: Defines or formats a text string used by the program.

        "group_technique_count": 0,  # Line 211: Defines or formats a text string used by the program.

        "technique_frequency": 0,  # Line 213: Defines or formats a text string used by the program.

        "group_software_count":  # Line 215: Starts a Python block; the following indented lines belong to this block.
            group_software_count,  # Line 216: This line performs the operation shown in the original code; the surrounding block determines its exact role.

        "group_campaign_count":  # Line 218: Starts a Python block; the following indented lines belong to this block.
            group_campaign_count,  # Line 219: This line performs the operation shown in the original code; the surrounding block determines its exact role.

        "software_support_count": 0,  # Line 221: Defines or formats a text string used by the program.

        "campaign_support_count": 0,  # Line 223: Defines or formats a text string used by the program.

        "label": 0  # Line 225: Labels the constructed undocumented pair as class 0, meaning negative in this experiment.
    })  # Line 226: This line performs the operation shown in the original code; the surrounding block determines its exact role.


negative = pd.DataFrame(  # Line 229: Converts the collected negative examples into a DataFrame.
    negative_rows  # Line 230: This line performs the operation shown in the original code; the surrounding block determines its exact role.
)  # Line 231: Closes the current function/import/configuration block.


# ============================================================
# 6. COMBINE ML DATA
# ============================================================

ml_data = pd.concat(  # Line 238: Combines positive and negative examples into the final ML dataset.
    [  # Line 239: This line performs the operation shown in the original code; the surrounding block determines its exact role.
        positive,  # Line 240: This line performs the operation shown in the original code; the surrounding block determines its exact role.
        negative  # Line 241: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    ],  # Line 242: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    ignore_index=True  # Line 243: Assigns the value on the right-hand side to the variable on the left-hand side.
)  # Line 244: Closes the current function/import/configuration block.


print(  # Line 247: Prints information to the console for the user or for debugging.
    "\nML Dataset:",  # Line 248: Defines or formats a text string used by the program.
    ml_data.shape  # Line 249: This line performs the operation shown in the original code; the surrounding block determines its exact role.
)  # Line 250: Closes the current function/import/configuration block.

print("\nLabels:")  # Line 252: Prints information to the console for the user or for debugging.
print(  # Line 253: Prints information to the console for the user or for debugging.
    ml_data["label"].value_counts()  # Line 254: This line performs the operation shown in the original code; the surrounding block determines its exact role.
)  # Line 255: Closes the current function/import/configuration block.


# ============================================================
# 7. TRAIN / TEST SPLIT
# ============================================================

X = ml_data[  # Line 262: Selects the feature columns as the model input matrix X.
    feature_columns  # Line 263: This line performs the operation shown in the original code; the surrounding block determines its exact role.
]  # Line 264: This line performs the operation shown in the original code; the surrounding block determines its exact role.

y = ml_data[  # Line 266: Selects the label column as the target vector y.
    "label"  # Line 267: Defines or formats a text string used by the program.
]  # Line 268: This line performs the operation shown in the original code; the surrounding block determines its exact role.


X_train, X_test, y_train, y_test = train_test_split(  # Line 271: Splits features and labels into training and testing subsets.
    X,  # Line 272: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    y,  # Line 273: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    test_size=0.20,  # Line 274: Reserves 20 percent of the examples for testing.
    random_state=42,  # Line 275: Uses a fixed random seed so the split/model result is reproducible.
    stratify=y  # Line 276: Preserves the class distribution across the training and testing sets.
)  # Line 277: Closes the current function/import/configuration block.


# ============================================================
# 8. RANDOM FOREST
# ============================================================

model = RandomForestClassifier(  # Line 284: Creates the Random Forest classifier used by the project.
    n_estimators=200,  # Line 285: Requests 200 decision trees in the Random Forest.
    random_state=42,  # Line 286: Uses a fixed random seed so the split/model result is reproducible.
    class_weight="balanced"  # Line 287: Automatically gives more weight to the smaller class during training.
)  # Line 288: Closes the current function/import/configuration block.


model.fit(  # Line 291: Trains the Random Forest using the training features and labels.
    X_train,  # Line 292: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    y_train  # Line 293: This line performs the operation shown in the original code; the surrounding block determines its exact role.
)  # Line 294: Closes the current function/import/configuration block.


# ============================================================
# 9. MODEL EVALUATION
# ============================================================

y_pred = model.predict(  # Line 301: Generates predicted class labels for the unseen test set.
    X_test  # Line 302: This line performs the operation shown in the original code; the surrounding block determines its exact role.
)  # Line 303: Closes the current function/import/configuration block.


print(  # Line 306: Prints information to the console for the user or for debugging.
    "\n================ MODEL RESULTS ================"  # Line 307: Assigns the value on the right-hand side to the variable on the left-hand side.
)  # Line 308: Closes the current function/import/configuration block.

print(  # Line 310: Prints information to the console for the user or for debugging.
    "Accuracy :",  # Line 311: Defines or formats a text string used by the program.
    round(  # Line 312: This line performs the operation shown in the original code; the surrounding block determines its exact role.
        accuracy_score(  # Line 313: Calculates the fraction of test examples classified correctly.
            y_test,  # Line 314: This line performs the operation shown in the original code; the surrounding block determines its exact role.
            y_pred  # Line 315: This line performs the operation shown in the original code; the surrounding block determines its exact role.
        ) * 100,  # Line 316: Converts the metric from a 0–1 fraction into a percentage.
        2  # Line 317: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    ),  # Line 318: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    "%"  # Line 319: Defines or formats a text string used by the program.
)  # Line 320: Closes the current function/import/configuration block.

print(  # Line 322: Prints information to the console for the user or for debugging.
    "Precision:",  # Line 323: Defines or formats a text string used by the program.
    round(  # Line 324: This line performs the operation shown in the original code; the surrounding block determines its exact role.
        precision_score(  # Line 325: Calculates how many predicted positive relationships were actually positive.
            y_test,  # Line 326: This line performs the operation shown in the original code; the surrounding block determines its exact role.
            y_pred,  # Line 327: This line performs the operation shown in the original code; the surrounding block determines its exact role.
            zero_division=0  # Line 328: Assigns the value on the right-hand side to the variable on the left-hand side.
        ) * 100,  # Line 329: Converts the metric from a 0–1 fraction into a percentage.
        2  # Line 330: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    ),  # Line 331: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    "%"  # Line 332: Defines or formats a text string used by the program.
)  # Line 333: Closes the current function/import/configuration block.

print(  # Line 335: Prints information to the console for the user or for debugging.
    "Recall   :",  # Line 336: Defines or formats a text string used by the program.
    round(  # Line 337: This line performs the operation shown in the original code; the surrounding block determines its exact role.
        recall_score(  # Line 338: Calculates how many actual positive relationships were successfully detected.
            y_test,  # Line 339: This line performs the operation shown in the original code; the surrounding block determines its exact role.
            y_pred,  # Line 340: This line performs the operation shown in the original code; the surrounding block determines its exact role.
            zero_division=0  # Line 341: Assigns the value on the right-hand side to the variable on the left-hand side.
        ) * 100,  # Line 342: Converts the metric from a 0–1 fraction into a percentage.
        2  # Line 343: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    ),  # Line 344: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    "%"  # Line 345: Defines or formats a text string used by the program.
)  # Line 346: Closes the current function/import/configuration block.

print(  # Line 348: Prints information to the console for the user or for debugging.
    "F1 Score :",  # Line 349: Defines or formats a text string used by the program.
    round(  # Line 350: This line performs the operation shown in the original code; the surrounding block determines its exact role.
        f1_score(  # Line 351: Calculates the harmonic mean of precision and recall.
            y_test,  # Line 352: This line performs the operation shown in the original code; the surrounding block determines its exact role.
            y_pred,  # Line 353: This line performs the operation shown in the original code; the surrounding block determines its exact role.
            zero_division=0  # Line 354: Assigns the value on the right-hand side to the variable on the left-hand side.
        ) * 100,  # Line 355: Converts the metric from a 0–1 fraction into a percentage.
        2  # Line 356: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    ),  # Line 357: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    "%"  # Line 358: Defines or formats a text string used by the program.
)  # Line 359: Closes the current function/import/configuration block.


# ============================================================
# 10. FEATURE IMPORTANCE
# ============================================================

print(  # Line 366: Prints information to the console for the user or for debugging.
    "\n================ FEATURE IMPORTANCE ================"  # Line 367: Assigns the value on the right-hand side to the variable on the left-hand side.
)  # Line 368: Closes the current function/import/configuration block.

for feature, importance in zip(  # Line 370: Loops through feature names and their learned Random Forest importance values.
    feature_columns,  # Line 371: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    model.feature_importances_  # Line 372: Reads the relative importance assigned to each feature by the trained forest.
):  # Line 373: Starts a Python block; the following indented lines belong to this block.

    print(  # Line 375: Prints information to the console for the user or for debugging.
        f"{feature} : {importance:.4f}"  # Line 376: Defines or formats a text string used by the program.
    )  # Line 377: Closes the current function/import/configuration block.


# ============================================================
# 11. CREATE TECHNIQUE LOOKUPS
# ============================================================

print(  # Line 384: Prints information to the console for the user or for debugging.
    "\nBuilding ATT&CK lookups..."  # Line 385: Defines or formats a text string used by the program.
)  # Line 386: Closes the current function/import/configuration block.


# ------------------------------------------------------------
# ID -> Name
# ------------------------------------------------------------

technique_id_to_name = {}  # Line 393: Creates a lookup dictionary from ATT&CK technique ID to technique name.


# ------------------------------------------------------------
# Name -> ID
# ------------------------------------------------------------

technique_name_to_id = {}  # Line 400: Creates a lookup dictionary from normalized technique name to ATT&CK ID.


for _, row in data[  # Line 403: Iterates through unique technique ID/name pairs to build both lookup dictionaries.
    [  # Line 404: This line performs the operation shown in the original code; the surrounding block determines its exact role.
        "technique_id",  # Line 405: Defines or formats a text string used by the program.
        "technique_name"  # Line 406: Defines or formats a text string used by the program.
    ]  # Line 407: This line performs the operation shown in the original code; the surrounding block determines its exact role.
].drop_duplicates().iterrows():  # Line 408: Starts a Python block; the following indented lines belong to this block.

    tid = str(  # Line 410: Converts the technique ID to clean text.
        row["technique_id"]  # Line 411: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    ).strip()  # Line 412: This line performs the operation shown in the original code; the surrounding block determines its exact role.

    name = str(  # Line 414: Converts the technique name to clean text.
        row["technique_name"]  # Line 415: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    ).strip()  # Line 416: This line performs the operation shown in the original code; the surrounding block determines its exact role.

    technique_id_to_name[  # Line 418: Stores the technique name under its ATT&CK ID.
        tid  # Line 419: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    ] = name  # Line 420: Assigns the value on the right-hand side to the variable on the left-hand side.

    technique_name_to_id[  # Line 422: Stores the ATT&CK ID under the normalized technique name.
        normalize(name)  # Line 423: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    ] = tid  # Line 424: Assigns the value on the right-hand side to the variable on the left-hand side.


# ============================================================
# 12. SMART TECHNIQUE SEARCH
# ============================================================

def find_technique(user_input):  # Line 431: Defines the smart technique-matching function used for user input.

    user_input = user_input.strip()  # Line 433: Assigns the value on the right-hand side to the variable on the left-hand side.

    normalized_input = normalize(  # Line 435: Normalizes the user-entered technique name before searching.
        user_input  # Line 436: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    )  # Line 437: Closes the current function/import/configuration block.

    # --------------------------------------------------------
    # Direct ID match
    # --------------------------------------------------------

    for tid in technique_id_to_name:  # Line 443: Checks all known ATT&CK IDs for a direct ID match.

        if normalized_input == normalize(tid):  # Line 445: Starts a Python block; the following indented lines belong to this block.

            return tid  # Line 447: Returns the matching ATT&CK technique ID.


    # --------------------------------------------------------
    # Exact full-name match
    # --------------------------------------------------------

    if normalized_input in technique_name_to_id:  # Line 454: Checks for an exact full technique-name match.

        return technique_name_to_id[  # Line 456: Stores the ATT&CK ID under the normalized technique name.
            normalized_input  # Line 457: This line performs the operation shown in the original code; the surrounding block determines its exact role.
        ]  # Line 458: This line performs the operation shown in the original code; the surrounding block determines its exact role.


    # --------------------------------------------------------
    # Search last part of sub-technique
    #
    # Example:
    # PowerShell
    #
    # matches:
    # Command and Scripting Interpreter: PowerShell
    # --------------------------------------------------------

    matches = []  # Line 471: Creates a list to collect possible partial-name matches.

    for name, tid in technique_name_to_id.items():  # Line 473: Searches all technique names for a matching or containing text.

        if (  # Line 475: Checks a condition and executes the indented block when it is true.
            normalized_input == name  # Line 476: Normalizes the user-entered technique name before searching.
            or  # Line 477: This line performs the operation shown in the original code; the surrounding block determines its exact role.
            normalized_input in name  # Line 478: This line performs the operation shown in the original code; the surrounding block determines its exact role.
        ):  # Line 479: Starts a Python block; the following indented lines belong to this block.

            matches.append(  # Line 481: Stores a matching technique name and ID candidate.
                (name, tid)  # Line 482: This line performs the operation shown in the original code; the surrounding block determines its exact role.
            )  # Line 483: Closes the current function/import/configuration block.


    if len(matches) == 1:  # Line 486: Accepts the result only when exactly one candidate matches.

        return matches[0][1]  # Line 488: Returns the calculated value from the current function.


    # --------------------------------------------------------
    # Special matching for names after ":"
    # --------------------------------------------------------

    for name, tid in technique_name_to_id.items():  # Line 495: Searches all technique names for a matching or containing text.

        if ":" in name:  # Line 497: Checks sub-technique names that contain a parent name and colon.

            last_part = name.split(  # Line 499: Extracts the text after the colon, such as PowerShell from a full sub-technique name.
                ":"  # Line 500: Defines or formats a text string used by the program.
            )[-1].strip()  # Line 501: This line performs the operation shown in the original code; the surrounding block determines its exact role.

            if normalized_input == last_part:  # Line 503: Starts a Python block; the following indented lines belong to this block.

                return tid  # Line 505: Returns the matching ATT&CK technique ID.


    # --------------------------------------------------------
    # Nothing found
    # --------------------------------------------------------

    return None  # Line 512: Returns no result when the user input cannot be matched to an ATT&CK technique.


# ============================================================
# 13. CASE INFORMATION
# ============================================================

print(  # Line 519: Prints information to the console for the user or for debugging.
    "\n================ CASE INFORMATION ================"  # Line 520: Assigns the value on the right-hand side to the variable on the left-hand side.
)  # Line 521: Closes the current function/import/configuration block.

case_id = input(  # Line 523: Asks the user for a case identifier used as report metadata.
    "\nEnter Case ID: "  # Line 524: Defines or formats a text string used by the program.
).strip()  # Line 525: This line performs the operation shown in the original code; the surrounding block determines its exact role.

observation_period = input(  # Line 527: Asks the user for the observation period used as report metadata.
    "Enter Observation Period: "  # Line 528: Defines or formats a text string used by the program.
).strip()  # Line 529: This line performs the operation shown in the original code; the surrounding block determines its exact role.


# ============================================================
# 14. TECHNIQUES INPUT
# ============================================================

print(  # Line 536: Prints information to the console for the user or for debugging.
    "\nEnter observed MITRE ATT&CK techniques"  # Line 537: Defines or formats a text string used by the program.
)  # Line 538: Closes the current function/import/configuration block.

print(  # Line 540: Prints information to the console for the user or for debugging.
    "You can enter technique names or IDs."  # Line 541: Defines or formats a text string used by the program.
)  # Line 542: Closes the current function/import/configuration block.

print(  # Line 544: Prints information to the console for the user or for debugging.
    "Example: PowerShell"  # Line 545: Defines or formats a text string used by the program.
)  # Line 546: Closes the current function/import/configuration block.

print(  # Line 548: Prints information to the console for the user or for debugging.
    "Example: T1059.001"  # Line 549: Defines or formats a text string used by the program.
)  # Line 550: Closes the current function/import/configuration block.

print(  # Line 552: Prints information to the console for the user or for debugging.
    "Type 'done' when finished.\n"  # Line 553: Defines or formats a text string used by the program.
)  # Line 554: Closes the current function/import/configuration block.


observed_techniques = []  # Line 557: Creates a list to store recognized ATT&CK technique IDs entered by the user.

observed_technique_names = []  # Line 559: Creates a list to store the readable names of recognized techniques.


while True:  # Line 562: Starts a loop that repeatedly accepts user input until done is entered.

    value = input(  # Line 564: Reads one technique/software/platform value from the user.
        "Technique: "  # Line 565: Defines or formats a text string used by the program.
    ).strip()  # Line 566: This line performs the operation shown in the original code; the surrounding block determines its exact role.

    if value.lower() == "done":  # Line 568: Checks whether the user wants to finish entering values.

        break  # Line 570: This line performs the operation shown in the original code; the surrounding block determines its exact role.

    if not value:  # Line 572: Ignores blank input instead of processing it.

        continue  # Line 574: Skips this group and moves to the next group.


    technique_id = find_technique(  # Line 577: Converts the user-entered technique name or ID into a canonical ATT&CK ID.
        value  # Line 578: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    )  # Line 579: Closes the current function/import/configuration block.


    if technique_id:  # Line 582: Checks whether the technique was successfully recognized.

        technique_name = (  # Line 584: Looks up the readable ATT&CK technique name from the recognized ID.
            technique_id_to_name[  # Line 585: Stores the technique name under its ATT&CK ID.
                technique_id  # Line 586: This line performs the operation shown in the original code; the surrounding block determines its exact role.
            ]  # Line 587: This line performs the operation shown in the original code; the surrounding block determines its exact role.
        )  # Line 588: Closes the current function/import/configuration block.

        if technique_id not in observed_techniques:  # Line 590: Checks whether the technique was successfully recognized.

            observed_techniques.append(  # Line 592: Adds the recognized ATT&CK ID to the case input list.
                technique_id  # Line 593: This line performs the operation shown in the original code; the surrounding block determines its exact role.
            )  # Line 594: Closes the current function/import/configuration block.

            observed_technique_names.append(  # Line 596: Adds the readable technique name to the case input list.
                technique_name  # Line 597: This line performs the operation shown in the original code; the surrounding block determines its exact role.
            )  # Line 598: Closes the current function/import/configuration block.

            print(  # Line 600: Prints information to the console for the user or for debugging.
                f"✓ Recognized: "  # Line 601: Defines or formats a text string used by the program.
                f"{technique_name} "  # Line 602: Defines or formats a text string used by the program.
                f"({technique_id})"  # Line 603: Defines or formats a text string used by the program.
            )  # Line 604: Closes the current function/import/configuration block.

    else:  # Line 606: Starts a Python block; the following indented lines belong to this block.

        print(  # Line 608: Prints information to the console for the user or for debugging.
            f"✗ Technique not found: "  # Line 609: Defines or formats a text string used by the program.
            f"{value}"  # Line 610: Defines or formats a text string used by the program.
        )  # Line 611: Closes the current function/import/configuration block.

        print(  # Line 613: Prints information to the console for the user or for debugging.
            "Try the full ATT&CK technique name "  # Line 614: Defines or formats a text string used by the program.
            "or technique ID."  # Line 615: Defines or formats a text string used by the program.
        )  # Line 616: Closes the current function/import/configuration block.


# ============================================================
# 15. STOP IF NO VALID TECHNIQUES
# ============================================================

if not observed_techniques:  # Line 623: Starts a Python block; the following indented lines belong to this block.

    print(  # Line 625: Prints information to the console for the user or for debugging.
        "\n================================================"  # Line 626: Assigns the value on the right-hand side to the variable on the left-hand side.
    )  # Line 627: Closes the current function/import/configuration block.

    print(  # Line 629: Prints information to the console for the user or for debugging.
        "NO VALID MITRE ATT&CK TECHNIQUES DETECTED"  # Line 630: Displays an error message when no valid technique was entered.
    )  # Line 631: Closes the current function/import/configuration block.

    print(  # Line 633: Prints information to the console for the user or for debugging.
        "Please enter at least one valid technique."  # Line 634: Defines or formats a text string used by the program.
    )  # Line 635: Closes the current function/import/configuration block.

    print(  # Line 637: Prints information to the console for the user or for debugging.
        "================================================"  # Line 638: Assigns the value on the right-hand side to the variable on the left-hand side.
    )  # Line 639: Closes the current function/import/configuration block.

    raise SystemExit  # Line 641: Stops the program because the required data structure is not available.


# ============================================================
# 16. SOFTWARE INPUT
# ============================================================

print(  # Line 648: Prints information to the console for the user or for debugging.
    "\nEnter observed software/tools"  # Line 649: Defines or formats a text string used by the program.
)  # Line 650: Closes the current function/import/configuration block.

print(  # Line 652: Prints information to the console for the user or for debugging.
    "Type 'done' when finished.\n"  # Line 653: Defines or formats a text string used by the program.
)  # Line 654: Closes the current function/import/configuration block.


observed_software = []  # Line 657: Creates a list for observed software/tool evidence.


# Get all software from integrated dataset
all_software = set()  # Line 661: Creates a set that will contain all known software names from the dataset.


for value in data[  # Line 664: Starts a loop that processes each item in the specified iterable.
    "related_software"  # Line 665: Defines or formats a text string used by the program.
].dropna():  # Line 666: Starts a Python block; the following indented lines belong to this block.

    all_software.update(  # Line 668: Adds normalized software names to the known-software set.
        normalize(x)  # Line 669: This line performs the operation shown in the original code; the surrounding block determines its exact role.
        for x in split_values(value)  # Line 670: Starts a loop that processes each item in the specified iterable.
    )  # Line 671: Closes the current function/import/configuration block.


while True:  # Line 674: Starts a loop that repeatedly accepts user input until done is entered.

    value = input(  # Line 676: Reads one technique/software/platform value from the user.
        "Software: "  # Line 677: Defines or formats a text string used by the program.
    ).strip()  # Line 678: This line performs the operation shown in the original code; the surrounding block determines its exact role.

    if value.lower() == "done":  # Line 680: Checks whether the user wants to finish entering values.

        break  # Line 682: This line performs the operation shown in the original code; the surrounding block determines its exact role.

    if not value:  # Line 684: Ignores blank input instead of processing it.

        continue  # Line 686: Skips this group and moves to the next group.

    observed_software.append(  # Line 688: Adds the user-entered software/tool to the case evidence list.
        value  # Line 689: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    )  # Line 690: Closes the current function/import/configuration block.

    if normalize(value) in all_software:  # Line 692: Checks whether the entered software is known in the integrated dataset.

        print(  # Line 694: Prints information to the console for the user or for debugging.
            f"✓ Software found: {value}"  # Line 695: Defines or formats a text string used by the program.
        )  # Line 696: Closes the current function/import/configuration block.

    else:  # Line 698: Starts a Python block; the following indented lines belong to this block.

        print(  # Line 700: Prints information to the console for the user or for debugging.
            f"⚠ Software not found in "  # Line 701: Defines or formats a text string used by the program.
            f"integrated dataset: {value}"  # Line 702: Defines or formats a text string used by the program.
        )  # Line 703: Closes the current function/import/configuration block.

        print(  # Line 705: Prints information to the console for the user or for debugging.
            "It will still be displayed as "  # Line 706: Defines or formats a text string used by the program.
            "case evidence."  # Line 707: Defines or formats a text string used by the program.
        )  # Line 708: Closes the current function/import/configuration block.


# ============================================================
# 17. PLATFORM INPUT
# ============================================================

print(  # Line 715: Prints information to the console for the user or for debugging.
    "\nEnter target platform(s)"  # Line 716: Defines or formats a text string used by the program.
)  # Line 717: Closes the current function/import/configuration block.

print(  # Line 719: Prints information to the console for the user or for debugging.
    "Type 'done' when finished.\n"  # Line 720: Defines or formats a text string used by the program.
)  # Line 721: Closes the current function/import/configuration block.


observed_platforms = []  # Line 724: Creates a list for observed target platforms.


while True:  # Line 727: Starts a loop that repeatedly accepts user input until done is entered.

    value = input(  # Line 729: Reads one technique/software/platform value from the user.
        "Platform: "  # Line 730: Defines or formats a text string used by the program.
    ).strip()  # Line 731: This line performs the operation shown in the original code; the surrounding block determines its exact role.

    if value.lower() == "done":  # Line 733: Checks whether the user wants to finish entering values.

        break  # Line 735: This line performs the operation shown in the original code; the surrounding block determines its exact role.

    if not value:  # Line 737: Ignores blank input instead of processing it.

        continue  # Line 739: Skips this group and moves to the next group.

    observed_platforms.append(  # Line 741: This line performs the operation shown in the original code; the surrounding block determines its exact role.
        value  # Line 742: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    )  # Line 743: Closes the current function/import/configuration block.


# ============================================================
# 18. CAMPAIGN INPUT
# ============================================================

print(  # Line 750: Prints information to the console for the user or for debugging.
    "\nEnter known/suspected campaign."  # Line 751: Defines or formats a text string used by the program.
)  # Line 752: Closes the current function/import/configuration block.

campaign_input = input(  # Line 754: Reads an optional known or suspected campaign from the user.
    "Campaign (or Unknown): "  # Line 755: Defines or formats a text string used by the program.
).strip()  # Line 756: This line performs the operation shown in the original code; the surrounding block determines its exact role.


if normalize(  # Line 759: Checks a condition and executes the indented block when it is true.
    campaign_input  # Line 760: This line performs the operation shown in the original code; the surrounding block determines its exact role.
) == "unknown":  # Line 761: Starts a Python block; the following indented lines belong to this block.

    campaign_input = ""  # Line 763: Assigns the value on the right-hand side to the variable on the left-hand side.


# ============================================================
# 19. PREDICTION
# ============================================================

print(  # Line 770: Prints information to the console for the user or for debugging.
    "\nGenerating threat-group relationships..."  # Line 771: Defines or formats a text string used by the program.
)  # Line 772: Closes the current function/import/configuration block.


results = []  # Line 775: Creates an empty list for candidate threat-group ranking results.


groups = data[  # Line 778: Builds a unique list of threat groups to evaluate.
    [  # Line 779: This line performs the operation shown in the original code; the surrounding block determines its exact role.
        "group_id",  # Line 780: Defines or formats a text string used by the program.
        "group_name"  # Line 781: Defines or formats a text string used by the program.
    ]  # Line 782: This line performs the operation shown in the original code; the surrounding block determines its exact role.
].drop_duplicates()  # Line 783: This line performs the operation shown in the original code; the surrounding block determines its exact role.


for _, group in groups.iterrows():  # Line 786: Evaluates every threat group as a candidate.

    group_id = group[  # Line 788: Gets the current candidate group ID.
        "group_id"  # Line 789: Defines or formats a text string used by the program.
    ]  # Line 790: This line performs the operation shown in the original code; the surrounding block determines its exact role.

    group_name = group[  # Line 792: Gets the current candidate group name.
        "group_name"  # Line 793: Defines or formats a text string used by the program.
    ]  # Line 794: This line performs the operation shown in the original code; the surrounding block determines its exact role.


    group_data = data[  # Line 797: Filters the dataset to records belonging to the current threat group.
        data["group_id"] == group_id  # Line 798: Assigns the value on the right-hand side to the variable on the left-hand side.
    ]  # Line 799: This line performs the operation shown in the original code; the surrounding block determines its exact role.


    # --------------------------------------------------------
    # Matching techniques
    # --------------------------------------------------------

    matching = group_data[  # Line 806: Filters the group records to techniques observed in the current case.
        group_data[  # Line 807: This line performs the operation shown in the original code; the surrounding block determines its exact role.
            "technique_id"  # Line 808: Defines or formats a text string used by the program.
        ].isin(  # Line 809: This line performs the operation shown in the original code; the surrounding block determines its exact role.
            observed_techniques  # Line 810: This line performs the operation shown in the original code; the surrounding block determines its exact role.
        )  # Line 811: Closes the current function/import/configuration block.
    ]  # Line 812: This line performs the operation shown in the original code; the surrounding block determines its exact role.


    matching_count = len(  # Line 815: Counts how many observed techniques match this group.
        matching  # Line 816: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    )  # Line 817: Closes the current function/import/configuration block.


    total_observed = len(  # Line 820: Counts how many valid techniques were observed in the case.
        observed_techniques  # Line 821: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    )  # Line 822: Closes the current function/import/configuration block.


    overlap_score = (  # Line 825: Calculates the fraction of observed techniques that match this group.
        matching_count /  # Line 826: This line performs the operation shown in the original code; the surrounding block determines its exact role.
        total_observed  # Line 827: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    )  # Line 828: Closes the current function/import/configuration block.


    # --------------------------------------------------------
    # ML feature values
    # --------------------------------------------------------

    if matching_count > 0:  # Line 835: Checks whether at least one observed technique matches the group.

        case_features = matching[  # Line 837: Builds case-level ML features from the matching group-technique rows.
            feature_columns  # Line 838: This line performs the operation shown in the original code; the surrounding block determines its exact role.
        ].mean()  # Line 839: Averages the matching feature values to create one feature vector for this candidate group.


    else:  # Line 842: Starts a Python block; the following indented lines belong to this block.

        case_features = pd.Series(  # Line 844: Creates an all-zero feature vector when there are no matching techniques.
            {  # Line 845: This line performs the operation shown in the original code; the surrounding block determines its exact role.
                column: 0  # Line 846: This line performs the operation shown in the original code; the surrounding block determines its exact role.
                for column in feature_columns  # Line 847: Loops through every ML feature so it can be converted to numeric form.
            }  # Line 848: This line performs the operation shown in the original code; the surrounding block determines its exact role.
        )  # Line 849: Closes the current function/import/configuration block.


        case_features[  # Line 852: This line performs the operation shown in the original code; the surrounding block determines its exact role.
            "group_software_count"  # Line 853: Defines or formats a text string used by the program.
        ] = group_data[  # Line 854: Assigns the value on the right-hand side to the variable on the left-hand side.
            "group_software_count"  # Line 855: Defines or formats a text string used by the program.
        ].iloc[0]  # Line 856: This line performs the operation shown in the original code; the surrounding block determines its exact role.


        case_features[  # Line 859: This line performs the operation shown in the original code; the surrounding block determines its exact role.
            "group_campaign_count"  # Line 860: Defines or formats a text string used by the program.
        ] = group_data[  # Line 861: Assigns the value on the right-hand side to the variable on the left-hand side.
            "group_campaign_count"  # Line 862: Defines or formats a text string used by the program.
        ].iloc[0]  # Line 863: This line performs the operation shown in the original code; the surrounding block determines its exact role.


    X_case = pd.DataFrame(  # Line 866: Converts the candidate feature values into the DataFrame shape expected by the model.
        [case_features],  # Line 867: This line performs the operation shown in the original code; the surrounding block determines its exact role.
        columns=feature_columns  # Line 868: Assigns the value on the right-hand side to the variable on the left-hand side.
    )  # Line 869: Closes the current function/import/configuration block.


    ml_score = model.predict_proba(  # Line 872: Gets the Random Forest score for the positive relationship class.
        X_case  # Line 873: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    )[0][1]  # Line 874: This line performs the operation shown in the original code; the surrounding block determines its exact role.


    # --------------------------------------------------------
    # Software evidence
    # --------------------------------------------------------

    software_matches = []  # Line 881: Creates a list to collect possible partial-name matches.


    group_software = set()  # Line 884: Creates a normalized set of software associated with the candidate group.


    for value in group_data[  # Line 887: Starts a loop that processes each item in the specified iterable.
        "related_software"  # Line 888: Defines or formats a text string used by the program.
    ].dropna():  # Line 889: Starts a Python block; the following indented lines belong to this block.

        group_software.update(  # Line 891: Adds the group’s known software names to the set.
            normalize(x)  # Line 892: This line performs the operation shown in the original code; the surrounding block determines its exact role.
            for x in split_values(value)  # Line 893: Starts a loop that processes each item in the specified iterable.
        )  # Line 894: Closes the current function/import/configuration block.


    for software in observed_software:  # Line 897: Starts a Python block; the following indented lines belong to this block.

        if normalize(  # Line 899: Checks a condition and executes the indented block when it is true.
            software  # Line 900: This line performs the operation shown in the original code; the surrounding block determines its exact role.
        ) in group_software:  # Line 901: Starts a Python block; the following indented lines belong to this block.

            software_matches.append(  # Line 903: Stores a matching technique name and ID candidate.
                software  # Line 904: This line performs the operation shown in the original code; the surrounding block determines its exact role.
            )  # Line 905: Closes the current function/import/configuration block.


    if observed_software:  # Line 908: Checks whether the case contains any software evidence.

        software_score = (  # Line 910: Calculates the fraction of observed software that matches the candidate group.
            len(software_matches) /  # Line 911: This line performs the operation shown in the original code; the surrounding block determines its exact role.
            len(observed_software)  # Line 912: This line performs the operation shown in the original code; the surrounding block determines its exact role.
        )  # Line 913: Closes the current function/import/configuration block.

    else:  # Line 915: Starts a Python block; the following indented lines belong to this block.

        software_score = 0  # Line 917: Calculates the fraction of observed software that matches the candidate group.


    # --------------------------------------------------------
    # Campaign evidence
    # --------------------------------------------------------

    campaign_match = False  # Line 924: Initializes the campaign-match flag as false.


    if campaign_input:  # Line 927: Only performs campaign matching when the user supplied a campaign.

        group_campaigns = set()  # Line 929: Creates a normalized set of campaigns associated with the candidate group.


        for value in group_data[  # Line 932: Starts a loop that processes each item in the specified iterable.
            "related_campaigns"  # Line 933: Defines or formats a text string used by the program.
        ].dropna():  # Line 934: Starts a Python block; the following indented lines belong to this block.

            group_campaigns.update(  # Line 936: Adds the group’s known campaigns to the campaign set.
                normalize(x)  # Line 937: This line performs the operation shown in the original code; the surrounding block determines its exact role.
                for x in split_values(value)  # Line 938: Starts a loop that processes each item in the specified iterable.
            )  # Line 939: Closes the current function/import/configuration block.


        if normalize(  # Line 942: Checks a condition and executes the indented block when it is true.
            campaign_input  # Line 943: This line performs the operation shown in the original code; the surrounding block determines its exact role.
        ) in group_campaigns:  # Line 944: Starts a Python block; the following indented lines belong to this block.

            campaign_match = True  # Line 946: Initializes the campaign-match flag as false.


    campaign_score = (  # Line 949: Converts the campaign match into a binary score: 1 for match, 0 otherwise.
        1 if campaign_match  # Line 950: This line performs the operation shown in the original code; the surrounding block determines its exact role.
        else 0  # Line 951: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    )  # Line 952: Closes the current function/import/configuration block.


    # --------------------------------------------------------
    # Platform evidence
    # --------------------------------------------------------

    platform_match_count = 0  # Line 959: Initializes the number of matching target platforms.


    if observed_platforms:  # Line 962: Starts a Python block; the following indented lines belong to this block.

        group_platforms = set()  # Line 964: Creates a normalized set of platforms associated with the candidate group.


        for value in group_data[  # Line 967: Starts a loop that processes each item in the specified iterable.
            "platforms"  # Line 968: Defines or formats a text string used by the program.
        ].dropna():  # Line 969: Starts a Python block; the following indented lines belong to this block.

            group_platforms.update(  # Line 971: Adds the candidate group’s known platforms to the set.
                normalize(x)  # Line 972: This line performs the operation shown in the original code; the surrounding block determines its exact role.
                for x in split_values(value)  # Line 973: Starts a loop that processes each item in the specified iterable.
            )  # Line 974: Closes the current function/import/configuration block.


        for platform in observed_platforms:  # Line 977: Starts a Python block; the following indented lines belong to this block.

            if normalize(  # Line 979: Checks a condition and executes the indented block when it is true.
                platform  # Line 980: This line performs the operation shown in the original code; the surrounding block determines its exact role.
            ) in group_platforms:  # Line 981: Starts a Python block; the following indented lines belong to this block.

                platform_match_count += 1  # Line 983: Increments the platform match count when a case platform is supported.


    if observed_platforms:  # Line 986: Starts a Python block; the following indented lines belong to this block.

        platform_score = (  # Line 988: Calculates the fraction of observed platforms that match the group.
            platform_match_count /  # Line 989: This line performs the operation shown in the original code; the surrounding block determines its exact role.
            len(observed_platforms)  # Line 990: This line performs the operation shown in the original code; the surrounding block determines its exact role.
        )  # Line 991: Closes the current function/import/configuration block.

    else:  # Line 993: Starts a Python block; the following indented lines belong to this block.

        platform_score = 0  # Line 995: Calculates the fraction of observed platforms that match the group.


    # --------------------------------------------------------
    # FINAL RELATIONSHIP SCORE
    # --------------------------------------------------------
    #
    # ML = 55%
    # Technique overlap = 25%
    # Software = 10%
    # Platform = 5%
    # Campaign = 5%
    #
    # --------------------------------------------------------

    final_score = (  # Line 1010: Starts calculating the final composite relationship-ranking score.

        0.55 * ml_score +  # Line 1012: Gives the Random Forest score a 55 percent weight.

        0.25 * overlap_score +  # Line 1014: Gives observed technique overlap a 25 percent weight.

        0.10 * software_score +  # Line 1016: Gives software evidence a 10 percent weight.

        0.05 * platform_score +  # Line 1018: Gives platform evidence a 5 percent weight.

        0.05 * campaign_score  # Line 1020: Gives campaign evidence a 5 percent weight.

    )  # Line 1022: Closes the current function/import/configuration block.


    results.append({  # Line 1025: Stores the candidate group’s score and supporting evidence.

        "group_id":  # Line 1027: Starts a Python block; the following indented lines belong to this block.
            group_id,  # Line 1028: This line performs the operation shown in the original code; the surrounding block determines its exact role.

        "group_name":  # Line 1030: Starts a Python block; the following indented lines belong to this block.
            group_name,  # Line 1031: This line performs the operation shown in the original code; the surrounding block determines its exact role.

        "score":  # Line 1033: Stores the final relationship score as a percentage.
            final_score * 100,  # Line 1034: Converts the metric from a 0–1 fraction into a percentage.

        "matching_count":  # Line 1036: Stores the number of matching techniques.
            matching_count,  # Line 1037: This line performs the operation shown in the original code; the surrounding block determines its exact role.

        "total_observed":  # Line 1039: Starts a Python block; the following indented lines belong to this block.
            total_observed,  # Line 1040: This line performs the operation shown in the original code; the surrounding block determines its exact role.

        "software_matches":  # Line 1042: Stores the matching software evidence.
            software_matches,  # Line 1043: This line performs the operation shown in the original code; the surrounding block determines its exact role.

        "platform_matches":  # Line 1045: Stores the number of matching platforms.
            platform_match_count,  # Line 1046: This line performs the operation shown in the original code; the surrounding block determines its exact role.

        "campaign_match":  # Line 1048: Stores whether the campaign matched.
            campaign_match,  # Line 1049: This line performs the operation shown in the original code; the surrounding block determines its exact role.

        "matching_techniques":  # Line 1051: Stores the names of the techniques that matched the candidate group.
            matching[  # Line 1052: This line performs the operation shown in the original code; the surrounding block determines its exact role.
                "technique_name"  # Line 1053: Defines or formats a text string used by the program.
            ].tolist()  # Line 1054: This line performs the operation shown in the original code; the surrounding block determines its exact role.

    })  # Line 1056: This line performs the operation shown in the original code; the surrounding block determines its exact role.


# ============================================================
# 20. SORT RESULTS
# ============================================================

results = sorted(  # Line 1063: Sorts all candidate groups by their relationship score.
    results,  # Line 1064: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    key=lambda x: x["score"],  # Line 1065: Uses the candidate score as the sorting key.
    reverse=True  # Line 1066: Sorts from highest score to lowest score.
)  # Line 1067: Closes the current function/import/configuration block.


top_results = results[:5]  # Line 1070: Keeps only the five highest-ranked threat groups for the report.


# ============================================================
# 21. REPORT
# ============================================================

print(  # Line 1077: Prints information to the console for the user or for debugging.
    "\n\n================================================"  # Line 1078: Assigns the value on the right-hand side to the variable on the left-hand side.
)  # Line 1079: Closes the current function/import/configuration block.

print(  # Line 1081: Prints information to the console for the user or for debugging.
    "             CYBER THREAT INTELLIGENCE REPORT"  # Line 1082: Defines or formats a text string used by the program.
)  # Line 1083: Closes the current function/import/configuration block.

print(  # Line 1085: Prints information to the console for the user or for debugging.
    "================================================"  # Line 1086: Assigns the value on the right-hand side to the variable on the left-hand side.
)  # Line 1087: Closes the current function/import/configuration block.


print(  # Line 1090: Prints information to the console for the user or for debugging.
    f"\nCase ID: {case_id}"  # Line 1091: Defines or formats a text string used by the program.
)  # Line 1092: Closes the current function/import/configuration block.


print(  # Line 1095: Prints information to the console for the user or for debugging.
    f"Observation Period: "  # Line 1096: Defines or formats a text string used by the program.
    f"{observation_period}"  # Line 1097: Defines or formats a text string used by the program.
)  # Line 1098: Closes the current function/import/configuration block.


# ------------------------------------------------------------
# Techniques
# ------------------------------------------------------------

print(  # Line 1105: Prints information to the console for the user or for debugging.
    "\nObserved Techniques:"  # Line 1106: Defines or formats a text string used by the program.
)  # Line 1107: Closes the current function/import/configuration block.


for i, technique in enumerate(  # Line 1110: Starts a loop that processes each item in the specified iterable.
    observed_technique_names,  # Line 1111: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    1  # Line 1112: This line performs the operation shown in the original code; the surrounding block determines its exact role.
):  # Line 1113: Starts a Python block; the following indented lines belong to this block.

    print(  # Line 1115: Prints information to the console for the user or for debugging.
        f"{i}. {technique}"  # Line 1116: Defines or formats a text string used by the program.
    )  # Line 1117: Closes the current function/import/configuration block.


# ------------------------------------------------------------
# Software
# ------------------------------------------------------------

print(  # Line 1124: Prints information to the console for the user or for debugging.
    "\nObserved Software/Tools:"  # Line 1125: Defines or formats a text string used by the program.
)  # Line 1126: Closes the current function/import/configuration block.


if observed_software:  # Line 1129: Checks whether the case contains any software evidence.

    for i, software in enumerate(  # Line 1131: Starts a loop that processes each item in the specified iterable.
        observed_software,  # Line 1132: This line performs the operation shown in the original code; the surrounding block determines its exact role.
        1  # Line 1133: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    ):  # Line 1134: Starts a Python block; the following indented lines belong to this block.

        print(  # Line 1136: Prints information to the console for the user or for debugging.
            f"{i}. {software}"  # Line 1137: Defines or formats a text string used by the program.
        )  # Line 1138: Closes the current function/import/configuration block.

else:  # Line 1140: Starts a Python block; the following indented lines belong to this block.

    print(  # Line 1142: Prints information to the console for the user or for debugging.
        "None provided"  # Line 1143: Defines or formats a text string used by the program.
    )  # Line 1144: Closes the current function/import/configuration block.


# ------------------------------------------------------------
# Platforms
# ------------------------------------------------------------

print(  # Line 1151: Prints information to the console for the user or for debugging.
    "\nTarget Platforms:"  # Line 1152: Defines or formats a text string used by the program.
)  # Line 1153: Closes the current function/import/configuration block.


if observed_platforms:  # Line 1156: Starts a Python block; the following indented lines belong to this block.

    for i, platform in enumerate(  # Line 1158: Starts a loop that processes each item in the specified iterable.
        observed_platforms,  # Line 1159: This line performs the operation shown in the original code; the surrounding block determines its exact role.
        1  # Line 1160: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    ):  # Line 1161: Starts a Python block; the following indented lines belong to this block.

        print(  # Line 1163: Prints information to the console for the user or for debugging.
            f"{i}. {platform}"  # Line 1164: Defines or formats a text string used by the program.
        )  # Line 1165: Closes the current function/import/configuration block.

else:  # Line 1167: Starts a Python block; the following indented lines belong to this block.

    print(  # Line 1169: Prints information to the console for the user or for debugging.
        "None provided"  # Line 1170: Defines or formats a text string used by the program.
    )  # Line 1171: Closes the current function/import/configuration block.


# ------------------------------------------------------------
# Campaign
# ------------------------------------------------------------

print(  # Line 1178: Prints information to the console for the user or for debugging.
    "\nCampaign:"  # Line 1179: Defines or formats a text string used by the program.
)  # Line 1180: Closes the current function/import/configuration block.


if campaign_input:  # Line 1183: Only performs campaign matching when the user supplied a campaign.

    print(  # Line 1185: Prints information to the console for the user or for debugging.
        campaign_input  # Line 1186: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    )  # Line 1187: Closes the current function/import/configuration block.

else:  # Line 1189: Starts a Python block; the following indented lines belong to this block.

    print(  # Line 1191: Prints information to the console for the user or for debugging.
        "Unknown"  # Line 1192: Defines or formats a text string used by the program.
    )  # Line 1193: Closes the current function/import/configuration block.


# ============================================================
# 22. PREDICTIONS
# ============================================================

print(  # Line 1200: Prints information to the console for the user or for debugging.
    "\n================ PREDICTIONS ================"  # Line 1201: Assigns the value on the right-hand side to the variable on the left-hand side.
)  # Line 1202: Closes the current function/import/configuration block.


for i, result in enumerate(  # Line 1205: Starts a loop that processes each item in the specified iterable.
    top_results,  # Line 1206: This line performs the operation shown in the original code; the surrounding block determines its exact role.
    1  # Line 1207: This line performs the operation shown in the original code; the surrounding block determines its exact role.
):  # Line 1208: Starts a Python block; the following indented lines belong to this block.

    print(  # Line 1210: Prints information to the console for the user or for debugging.
        f"\n{i}. "  # Line 1211: Defines or formats a text string used by the program.
        f"{result['group_name']} "  # Line 1212: Defines or formats a text string used by the program.
        f"({result['group_id']})"  # Line 1213: Defines or formats a text string used by the program.
    )  # Line 1214: Closes the current function/import/configuration block.


    print(  # Line 1217: Prints information to the console for the user or for debugging.
        f"   Relationship Score: "  # Line 1218: Defines or formats a text string used by the program.
        f"{result['score']:.2f}%"  # Line 1219: Defines or formats a text string used by the program.
    )  # Line 1220: Closes the current function/import/configuration block.


    print(  # Line 1223: Prints information to the console for the user or for debugging.
        f"   Matching Techniques: "  # Line 1224: Defines or formats a text string used by the program.
        f"{result['matching_count']}/"  # Line 1225: Defines or formats a text string used by the program.
        f"{result['total_observed']}"  # Line 1226: Defines or formats a text string used by the program.
    )  # Line 1227: Closes the current function/import/configuration block.


    # Technique evidence
    if result[  # Line 1231: Checks a condition and executes the indented block when it is true.
        "matching_techniques"  # Line 1232: Defines or formats a text string used by the program.
    ]:  # Line 1233: Starts a Python block; the following indented lines belong to this block.

        print(  # Line 1235: Prints information to the console for the user or for debugging.
            "   Technique Evidence: "  # Line 1236: Defines or formats a text string used by the program.
            +  # Line 1237: This line performs the operation shown in the original code; the surrounding block determines its exact role.
            ", ".join(  # Line 1238: Defines or formats a text string used by the program.
                result[  # Line 1239: This line performs the operation shown in the original code; the surrounding block determines its exact role.
                    "matching_techniques"  # Line 1240: Defines or formats a text string used by the program.
                ]  # Line 1241: This line performs the operation shown in the original code; the surrounding block determines its exact role.
            )  # Line 1242: Closes the current function/import/configuration block.
        )  # Line 1243: Closes the current function/import/configuration block.

    else:  # Line 1245: Starts a Python block; the following indented lines belong to this block.

        print(  # Line 1247: Prints information to the console for the user or for debugging.
            "   Technique Evidence: None"  # Line 1248: Defines or formats a text string used by the program.
        )  # Line 1249: Closes the current function/import/configuration block.


    # Software evidence
    print(  # Line 1253: Prints information to the console for the user or for debugging.
        f"   Software Evidence: "  # Line 1254: Defines or formats a text string used by the program.
        f"{len(result['software_matches'])}/"  # Line 1255: Defines or formats a text string used by the program.
        f"{len(observed_software)}"  # Line 1256: Defines or formats a text string used by the program.
    )  # Line 1257: Closes the current function/import/configuration block.


    if result[  # Line 1260: Checks a condition and executes the indented block when it is true.
        "software_matches"  # Line 1261: Defines or formats a text string used by the program.
    ]:  # Line 1262: Starts a Python block; the following indented lines belong to this block.

        print(  # Line 1264: Prints information to the console for the user or for debugging.
            "   Matched Software: "  # Line 1265: Defines or formats a text string used by the program.
            +  # Line 1266: This line performs the operation shown in the original code; the surrounding block determines its exact role.
            ", ".join(  # Line 1267: Defines or formats a text string used by the program.
                result[  # Line 1268: This line performs the operation shown in the original code; the surrounding block determines its exact role.
                    "software_matches"  # Line 1269: Defines or formats a text string used by the program.
                ]  # Line 1270: This line performs the operation shown in the original code; the surrounding block determines its exact role.
            )  # Line 1271: Closes the current function/import/configuration block.
        )  # Line 1272: Closes the current function/import/configuration block.


    # Platform evidence
    print(  # Line 1276: Prints information to the console for the user or for debugging.
        f"   Platform Evidence: "  # Line 1277: Defines or formats a text string used by the program.
        f"{result['platform_matches']}/"  # Line 1278: Defines or formats a text string used by the program.
        f"{len(observed_platforms)}"  # Line 1279: Defines or formats a text string used by the program.
    )  # Line 1280: Closes the current function/import/configuration block.


    # Campaign evidence
    if campaign_input:  # Line 1284: Only performs campaign matching when the user supplied a campaign.

        print(  # Line 1286: Prints information to the console for the user or for debugging.
            "   Campaign Evidence: "  # Line 1287: Defines or formats a text string used by the program.
            +  # Line 1288: This line performs the operation shown in the original code; the surrounding block determines its exact role.
            (  # Line 1289: This line performs the operation shown in the original code; the surrounding block determines its exact role.
                "MATCH"  # Line 1290: Defines or formats a text string used by the program.
                if result[  # Line 1291: Checks a condition and executes the indented block when it is true.
                    "campaign_match"  # Line 1292: Defines or formats a text string used by the program.
                ]  # Line 1293: This line performs the operation shown in the original code; the surrounding block determines its exact role.
                else  # Line 1294: This line performs the operation shown in the original code; the surrounding block determines its exact role.
                "No match"  # Line 1295: Defines or formats a text string used by the program.
            )  # Line 1296: Closes the current function/import/configuration block.
        )  # Line 1297: Closes the current function/import/configuration block.


# ============================================================
# 23. FINAL ASSESSMENT
# ============================================================

best = top_results[0]  # Line 1304: Selects the highest-ranked candidate group.


score = best[  # Line 1307: Reads the best candidate’s relationship score.
    "score"  # Line 1308: Defines or formats a text string used by the program.
]  # Line 1309: This line performs the operation shown in the original code; the surrounding block determines its exact role.


if score >= 80:  # Line 1312: Assigns HIGH confidence when the ranking score is at least 80.

    confidence = "HIGH"  # Line 1314: Stores the HIGH confidence label.

elif score >= 60:  # Line 1316: Assigns MEDIUM confidence when the score is between 60 and 79.99.

    confidence = "MEDIUM"  # Line 1318: Stores the MEDIUM confidence label.

else:  # Line 1320: Starts a Python block; the following indented lines belong to this block.

    confidence = "LOW"  # Line 1322: Stores the LOW confidence label.


print(  # Line 1325: Prints information to the console for the user or for debugging.
    "\n================ FINAL ASSESSMENT ================"  # Line 1326: Assigns the value on the right-hand side to the variable on the left-hand side.
)  # Line 1327: Closes the current function/import/configuration block.


print(  # Line 1330: Prints information to the console for the user or for debugging.
    f"\nMost likely related group: "  # Line 1331: Prints the highest-ranked potentially related threat group.
    f"{best['group_name']}"  # Line 1332: Defines or formats a text string used by the program.
)  # Line 1333: Closes the current function/import/configuration block.


print(  # Line 1336: Prints information to the console for the user or for debugging.
    f"Relationship Score: "  # Line 1337: Defines or formats a text string used by the program.
    f"{score:.2f}%"  # Line 1338: Defines or formats a text string used by the program.
)  # Line 1339: Closes the current function/import/configuration block.


print(  # Line 1342: Prints information to the console for the user or for debugging.
    f"Confidence Level: "  # Line 1343: Prints the qualitative confidence category derived from the score threshold.
    f"{confidence}"  # Line 1344: Defines or formats a text string used by the program.
)  # Line 1345: Closes the current function/import/configuration block.


print(  # Line 1348: Prints information to the console for the user or for debugging.
    "\nNOTE: This is a behavioral "  # Line 1349: Defines or formats a text string used by the program.
    "relationship-ranking result,"  # Line 1350: Defines or formats a text string used by the program.
)  # Line 1351: Closes the current function/import/configuration block.


print(  # Line 1354: Prints information to the console for the user or for debugging.
    "not definitive threat-actor attribution."  # Line 1355: Explicitly states that the output is a behavioral ranking, not definitive attribution.
)  # Line 1356: Closes the current function/import/configuration block.


print(  # Line 1359: Prints information to the console for the user or for debugging.
    "\n================================================"  # Line 1360: Assigns the value on the right-hand side to the variable on the left-hand side.
)  # Line 1361: Closes the current function/import/configuration block.



Loading integrated dataset...

Dataset Shape: (4546, 26)

Columns:
['group_id', 'group_name', 'group_domain', 'group_version', 'technique_id', 'technique_name', 'technique_domain', 'technique_version', 'tactics', 'platforms', 'is_subtechnique', 'parent_technique_id', 'relationship_type', 'mapping_description', 'group_technique_count', 'group_software_count', 'group_campaign_count', 'technique_frequency', 'technique_group_count', 'software_support_count', 'software_support_ratio', 'related_software', 'campaign_support_count', 'campaign_support_ratio', 'related_campaigns', 'relationship_in_relationships']

After cleaning: (4546, 26)

Creating negative samples...

ML Dataset: (4716, 9)

Labels:
label
1    4546
0     170
Name: count, dtype: int64

================ MODEL RESULTS ================
Accuracy : 100.0 %
Precision: 100.0 %
Recall   : 100.0 %
F1 Score : 100.0 %

================ FEATURE IMPORTANCE ================
group_technique_count : 0.4776
technique_frequency : 0.4309
group_s


Enter Case ID:  101
Enter Observation Period:  2026



Enter observed MITRE ATT&CK techniques
You can enter technique names or IDs.
Example: PowerShell
Example: T1059.001
Type 'done' when finished.



Technique:  PowerShell


✓ Recognized: Command and Scripting Interpreter: PowerShell (T1059.001)


...

✓ Software found: Mimikatz


Software:   PsExec


✓ Software found: PsExec


Software:  done



Enter target platform(s)
Type 'done' when finished.

